In [ ]:
#@title Cell 54.1 — Scientific aim, definitions and interpretation
from IPython.display import display, Markdown

display(Markdown(r"""
# NB54 — BMC-N MIC breakpoint-exceedance probabilities (v0.1)

**Question:** For each previously unseen held-out *E. coli* pathogen, how likely is its MIC to exceed the antibiotic-specific EUCAST resistance cutoff?

\[
P(Y_{ia}>\log_2 R_a\mid\widehat{y}_{ia},\mathcal{D}^{\mathrm{train}}_{a,k})
\]

Here, \(Y_{ia}\) is log₂(MIC); \(R_a\) is the EUCAST MIC resistance cutoff in mg/L; \(\widehat{y}_{ia}\) is the **unchanged** held-out Model C prediction; and the BMC-N predictive error distribution is fitted using **training-only** cross-fitted residuals for antibiotic \(a\) and outer fold \(k\). The event is strictly MIC **>** the cutoff, in accordance with NB53.

### Pre-specified analysis
- Reconstruct the BMC-N mixture CDF using the **same exact grid/prior/numerical method as NB41**. NB41 saved predictive quantiles but did **not** save the mixture weights. A normal approximation from the 95% limits is **not** acceptable.
- Validate the reconstructed posterior against NB41's held-out interval bounds and posterior summaries **before** calculating probabilities.
- Use the **18 scenario-specific EUCAST v16.1 cutoffs** audited in NB53; do not classify the eight antibiotics without applicable clinical MIC breakpoints.
- For all held-out records in each applicable antibiotic, compute probabilities **without consulting the observed MIC**. Use the observed MIC only subsequently to evaluate the exceedance outcome.
- Assess calibration (predicted versus observed exceedance frequency) and accuracy (Brier score, log loss); show high-probability subsets (≥0.90) and low-probability subsets (≤0.10) without calling these clinically validated decisions.

**Clinical caveats:** Infection indication/specimen type are unknown, so indication-restricted EUCAST thresholds define **threshold-exceedance outcomes**, not verified resistance classifications for every patient. TMP–SMX MIC coding and piperacillin–tazobactam's area of technical uncertainty require caution (see NB53). Predictions are not a substitute for laboratory susceptibility testing.

Outputs are new NB54 files. **No existing results or manuscript files are modified.**
"""))

In [ ]:
#@title Cell 54.2 — Mount Drive; locate exact NB40, NB41 and NB53 outputs
import os, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.special import logsumexp, ndtr
from scipy.stats import norm
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
except ImportError:
    pass  # Local Jupyter: set BMC_PROJECT_ROOT to the RefMIC folder.

ROOT = Path(os.environ.get('BMC_PROJECT_ROOT', '/content/drive/MyDrive/RefMIC'))
if not ROOT.is_dir():
    raise FileNotFoundError(f'RefMIC folder not found: {ROOT}. Set BMC_PROJECT_ROOT if running locally.')
RESULTS = ROOT / '03_Results'
OUT = RESULTS / '54_BMC_Breakpoint_Exceedance_Probability'
OUT.mkdir(parents=True, exist_ok=True)

FILENAMES = {
    'calibration': ('40_BMC_Training_Only_Calibration', '40_BMC_training_only_inner_predictions_all_outer_folds.csv.gz'),
    'predictions': ('41_BMC_N_Pathogen_Out_Validation', '41_BMC_N_independent_outer_test_predictions.csv.gz'),
    'posterior_nb41': ('41_BMC_N_Pathogen_Out_Validation', '41_BMC_N_training_only_posterior_by_antibiotic_and_fold.csv'),
    'breakpoints_nb53': ('53_BMC_Resistance_Threshold_Coverage', '53_Supplementary_Resistance_Coverage_18_Antibiotics.csv'),
}

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def locate_prior_output(folder, filename):
    expected = RESULTS / folder / filename
    if expected.is_file():
        return expected
    found = []
    for directory, dirs, names in os.walk(ROOT):
        dirs[:] = [d for d in dirs if d not in {'.Trashes', '.git', '__pycache__', 'node_modules'}]
        if filename in names:
            found.append(Path(directory) / filename)
    if not found:
        raise FileNotFoundError(f'Missing: {filename}\nExpected: {expected}\nRun NB40 Cell 40.7, NB41 Cell 41.8 or NB53 Cell 53.7 as needed.')
    if len(found) > 1 and len({file_hash(p) for p in found}) > 1:
        raise ValueError('Multiple non-identical copies found for ' + filename + '\n' + '\n'.join(map(str, found)))
    return sorted(found, key=lambda p:(len(p.parts), str(p)))[0]

PATHS = {k: locate_prior_output(*spec) for k,spec in FILENAMES.items()}
for label,p in PATHS.items():
    print(f'{label:18s}: {p}')
print('NB54 outputs:', OUT)

In [ ]:
#@title Cell 54.3 — Load and audit 18 threshold scenarios and held-out observations
thresholds = pd.read_csv(PATHS['breakpoints_nb53'])
test = pd.read_csv(PATHS['predictions'], compression='gzip')
nb41_post = pd.read_csv(PATHS['posterior_nb41'])

NEEDED_THRESH = {'antibiotic', 'r_cutoff_mg_L_exclusive', 'clinical_scenario',
                  'n_tested', 'n_above_r_cutoff', 'interpretive_note'}
NEEDED_TEST = {'interaction_row','biosample','antibiotic','outer_fold','observed_log2_mic',
               'model_c_outer_test_predicted_log2_mic','bmc_n_lower95_log2','bmc_n_upper95_log2'}
NEEDED_POST = {'antibiotic','outer_fold','n_calibration','posterior_mean_b_log2',
               'posterior_mean_sigma_log2','predictive_residual_lower95_log2',
               'predictive_residual_upper95_log2'}
for label, df, required in [('NB53 scenarios',thresholds,NEEDED_THRESH),
                            ('NB41 test records',test,NEEDED_TEST),
                            ('NB41 posterior summary',nb41_post,NEEDED_POST)]:
    missing = required-set(df.columns)
    if missing:
        raise ValueError(f'{label}: missing {sorted(missing)}')

for tab in (thresholds,test,nb41_post):
    tab['antibiotic'] = tab['antibiotic'].astype(str).str.strip().str.lower()
assert len(thresholds)==18 and thresholds.antibiotic.is_unique
assert (len(test),test.biosample.nunique(),test.antibiotic.nunique())==(50460,9058,26)
assert len(nb41_post)==130 and not nb41_post.duplicated(['antibiotic','outer_fold']).any()
assert not test.interaction_row.duplicated().any() and not test.duplicated(['biosample','antibiotic']).any()
assert (test.groupby('biosample').outer_fold.nunique()==1).all()
assert set(test.outer_fold)==set(range(1,6))
assert set(thresholds.antibiotic).issubset(set(test.antibiotic))
assert (thresholds.r_cutoff_mg_L_exclusive>0).all()

use = test.loc[test.antibiotic.isin(thresholds.antibiotic)].copy()
use = use.merge(thresholds[['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note']],
                how='left',on='antibiotic',validate='many_to_one')
use['cutoff_log2_mic'] = np.log2(use.r_cutoff_mg_L_exclusive.to_numpy(float))
# Observed outcome is retained strictly for subsequent evaluation, not for BMC fitting.
use['observed_exceeds_r'] = use.observed_log2_mic.to_numpy(float) > use.cutoff_log2_mic.to_numpy(float)
assert len(use)==int(thresholds.n_tested.sum())

for t in [test,use]:
    vals=[c for c in ['observed_log2_mic','model_c_outer_test_predicted_log2_mic',
                     'bmc_n_lower95_log2','bmc_n_upper95_log2'] if c in t]
    assert np.isfinite(t[vals].to_numpy(dtype=float)).all()

counts=use.groupby('antibiotic').agg(n_tested=('interaction_row','size'),n_above_r_cutoff=('observed_exceeds_r','sum')).reset_index()
check=thresholds[['antibiotic','n_tested','n_above_r_cutoff']].merge(counts,on='antibiotic',suffixes=('_nb53','_nb54'),validate='one_to_one')
for col in ('n_tested','n_above_r_cutoff'):
    if not (check[f'{col}_nb53'].to_numpy()==check[f'{col}_nb54'].to_numpy()).all():
        raise ValueError(f'NB54 cannot reproduce NB53: {col}')
assert len(check)==18
print(f'Validated {len(use):,} independent held-out interactions in 18 breakpoint scenarios.')
print('NB53 positive counts agree exactly for all 18 antibiotics.')
display(check.sort_values('antibiotic'))

In [ ]:
#@title Cell 54.4 — Reconstruct exact NB41 BMC-N posterior mixture from training-only errors
# This deliberately matches Notebook 41 Cell 41.4 (no Gaussian approximation to intervals).
SIGMA_GRID = np.exp(np.linspace(np.log(0.01), np.log(16.0), 4501))
LOG_SIGMA = np.log(SIGMA_GRID)
SIGMA2 = SIGMA_GRID**2
PRIOR_RATE = 0.5

# Read NB40 training-only inner cross-fitted residuals (not held-out test labels).
calib = pd.read_csv(PATHS['calibration'], compression='gzip',
                    usecols=['outer_fold_for_calibration','antibiotic','training_residual_log2',
                             'interaction_row','biosample'])
calib.antibiotic = calib.antibiotic.astype(str).str.strip().str.lower()
assert len(calib)==201840 and set(calib.outer_fold_for_calibration)==set(range(1,6))
assert not calib.duplicated(['outer_fold_for_calibration','interaction_row']).any()
assert np.isfinite(calib.training_residual_log2.to_numpy(float)).all()

# Assert strict pathogen-out train/test separation for every fold.
for fold in range(1,6):
    train_ids = set(calib.loc[calib.outer_fold_for_calibration.eq(fold),'biosample'])
    test_ids = set(test.loc[test.outer_fold.eq(fold),'biosample'])
    assert not train_ids.intersection(test_ids), f'Fold {fold}: training/test BioSample leakage'


def fit_bmc_n_mixture(errors):
    e=np.asarray(errors,dtype=np.float64)
    n=len(e)
    if n<15 or not np.isfinite(e).all():
        raise ValueError('Calibration fold has fewer than 15 finite residuals')
    mean=float(e.mean())
    sse=float(np.sum((e-mean)**2))
    log_density=(-(n-1)*LOG_SIGMA - 0.5*np.log(SIGMA2+n)
                 - sse/(2*SIGMA2) - n*mean**2/(2*(SIGMA2+n))
                 - PRIOR_RATE*SIGMA_GRID + LOG_SIGMA)
    weights=np.exp(log_density-logsumexp(log_density))
    edge_mass=float(weights[:5].sum()+weights[-5:].sum())
    if edge_mass>1e-8:
        raise ValueError(f'BMC-N posterior grid truncated: endpoint mass {edge_mass}')
    # Integrate b analytically given sigma, matching NB41.
    means=n*mean/(n+SIGMA2)
    var_b=SIGMA2/(n+SIGMA2)
    predictive_sds=np.sqrt(SIGMA2+var_b)
    return dict(weights=weights, means=means, predictive_sds=predictive_sds,
                n=n, mean_b=float(np.dot(weights,means)),
                mean_sigma=float(np.dot(weights,SIGMA_GRID)), endpoint_mass=edge_mass)


def residual_cdf(values, fit, chunk_size=128):
    """Exact NB41 normal-mixture CDF at log2 residual values (chunked to cap RAM)."""
    x=np.atleast_1d(np.asarray(values,dtype=np.float64))
    result=np.empty(len(x),dtype=np.float64)
    for i in range(0,len(x),chunk_size):
        block=x[i:i+chunk_size,None]
        cdf=ndtr((block-fit['means'][None,:])/fit['predictive_sds'][None,:])
        result[i:i+chunk_size]=cdf@fit['weights']
    return result

# Important: The fitted posterior depends ONLY on NB40 training residuals.
FITS={}
audit_rows=[]
for (fold, abx), t in use.groupby(['outer_fold','antibiotic'],sort=True):
    tr=calib.loc[calib.outer_fold_for_calibration.eq(fold) & calib.antibiotic.eq(abx)]
    fit=fit_bmc_n_mixture(tr.training_residual_log2.to_numpy())
    ref=nb41_post.loc[nb41_post.outer_fold.eq(fold) & nb41_post.antibiotic.eq(abx)]
    if len(ref)!=1:
        raise ValueError(f'Missing NB41 posterior for fold={fold}, antibiotic={abx}')
    ref=ref.iloc[0]
    if fit['n']!=int(ref.n_calibration):
        raise ValueError('NB41 calibration sample count mismatch')
    b_diff=abs(fit['mean_b']-float(ref.posterior_mean_b_log2))
    s_diff=abs(fit['mean_sigma']-float(ref.posterior_mean_sigma_log2))
    cdfs=residual_cdf(np.array([ref.predictive_residual_lower95_log2,
                                ref.predictive_residual_upper95_log2]),fit)
    quantile_max_error=max(abs(cdfs[0]-.025),abs(cdfs[1]-.975))
    # These strict checks prevent a silent mismatch between NB41 and NB54 methods.
    if b_diff>1e-7 or s_diff>1e-7 or quantile_max_error>1e-6:
        raise ValueError(f'NB41 mixture reproduction failed {abx}, fold {fold}: '
                         f'b diff={b_diff}, sigma diff={s_diff}, quantile-CDF error={quantile_max_error}')
    FITS[(int(fold),str(abx))]=fit
    audit_rows.append({'outer_fold':int(fold),'antibiotic':abx,'n_calibration':fit['n'],
                       'posterior_mean_b_abs_diff':b_diff,
                       'posterior_mean_sigma_abs_diff':s_diff,
                       'lower95_reproduced_cdf':cdfs[0],
                       'upper95_reproduced_cdf':cdfs[1],
                       'max_quantile_cdf_error':quantile_max_error,
                       'sigma_grid_endpoint_mass':fit['endpoint_mass']})

posterior_audit=pd.DataFrame(audit_rows)
assert len(FITS)==18*5
print(f'All {len(FITS)} BMC-N antibiotic × fold fits reproduce NB41 posterior mean and 95% predictive limits.')
print('Max difference in expected CDF at nominal 2.5% / 97.5% limits:',posterior_audit.max_quantile_cdf_error.max())
display(posterior_audit.sort_values('max_quantile_cdf_error',ascending=False).head(8).round(9))

In [ ]:
#@title Cell 54.5 — Calculate held-out predictive probability of exceeding MIC > R
# For a held-out point prediction yhat and log2 resistance cutoff r:
# P(Y>r|training) = 1 - sum_j w_j * Phi((r-yhat-m_j)/s_j).
# Only prediction, cutoff, and training-only posterior appear in this computation.

use['prob_mic_exceeds_r'] = np.nan
for (fold, abx), t in use.groupby(['outer_fold','antibiotic'],sort=True):
    fit=FITS[(int(fold),str(abx))]
    residual_threshold=t.cutoff_log2_mic.to_numpy(float) - t.model_c_outer_test_predicted_log2_mic.to_numpy(float)
    values=1.0-residual_cdf(residual_threshold,fit,chunk_size=128)
    use.loc[t.index,'prob_mic_exceeds_r']=values

p=use.prob_mic_exceeds_r.to_numpy(float)
assert np.isfinite(p).all() and ((0<=p)&(p<=1)).all(), 'Invalid Bayesian probabilities'
assert not use.interaction_row.duplicated().any()
use['prediction_class_p_ge_90']=use.prob_mic_exceeds_r>=.90
use['prediction_class_p_le_10']=use.prob_mic_exceeds_r<=.10
use['probability_bin']=pd.cut(use.prob_mic_exceeds_r,
                              bins=[0,.1,.2,.3,.4,.5,.6,.7,.8,.9,1],
                              labels=[f'{x/10:.1f}–{(x+1)/10:.1f}' for x in range(10)],
                              include_lowest=True)
print('Held-out probabilities computed:',len(use))
print('Probability range:',float(np.min(p)),float(np.max(p)))
display(use[['biosample','antibiotic','outer_fold','model_c_outer_test_predicted_log2_mic',
             'r_cutoff_mg_L_exclusive','prob_mic_exceeds_r','observed_exceeds_r']].head(10).round(4))

In [ ]:
#@title Cell 54.6 — Evaluate calibration and predictive performance by antibiotic
# Models are trained unconditionally on the MIC error distribution; labels are ONLY for evaluation.
# Brier score and log loss are proper scoring rules for binary exceedance probabilities.
# AUC/AP require both outcome classes and are descriptive in small or imbalanced groups.

EPS=1e-12

def evaluate_binary(t):
    y=t.observed_exceeds_r.to_numpy(dtype=int)
    q=t.prob_mic_exceeds_r.to_numpy(float)
    q_safe=np.clip(q,EPS,1-EPS)
    n=len(y)
    n_pos=int(y.sum())
    n_neg=n-n_pos
    base=float(y.mean())
    score=float(np.mean((q-y)**2))
    constant_base_brier=base*(1-base)  # retrospective evaluation reference, NOT a prediction model
    auc=float(roc_auc_score(y,q)) if (n_pos>0 and n_neg>0) else np.nan
    ap=float(average_precision_score(y,q)) if n_pos>0 else np.nan
    high=(q>=.9)
    low=(q<=.1)
    return pd.Series({
        'n_tested':n,'n_above_r':n_pos,'n_not_above_r':n_neg,
        'observed_exceedance_rate':base,
        'mean_predicted_exceedance_probability':float(q.mean()),
        'mean_prediction_minus_observed_rate':float(q.mean()-base),
        'brier_score':score,
        'reference_constant_observed_prevalence_brier':constant_base_brier,
        'brier_skill_vs_retrospective_prevalence':float(1-score/constant_base_brier) if constant_base_brier>0 else np.nan,
        'log_loss':float(-np.mean(y*np.log(q_safe)+(1-y)*np.log(1-q_safe))),
        'auroc':auc,'average_precision':ap,
        'n_prob_ge_90':int(high.sum()),
        'n_exceed_r_among_prob_ge_90':int(y[high].sum()),
        'observed_rate_among_prob_ge_90':float(y[high].mean()) if high.any() else np.nan,
        'n_prob_le_10':int(low.sum()),
        'n_not_exceed_r_among_prob_le_10':int((1-y[low]).sum()),
        'observed_nonexceed_rate_among_prob_le_10':float((1-y[low]).mean()) if low.any() else np.nan,
        'has_30_positive_and_30_negative':bool(n_pos>=30 and n_neg>=30)
    })

per_antibiotic = pd.DataFrame([{'antibiotic':abx,**evaluate_binary(t).to_dict()}
                               for abx,t in use.groupby('antibiotic',sort=True)])
per_antibiotic = per_antibiotic.merge(
    thresholds[['antibiotic','r_cutoff_mg_L_exclusive','clinical_scenario','interpretive_note']],
    on='antibiotic',how='left',validate='one_to_one')
all_18 = evaluate_binary(use).to_frame().T
all_18.insert(0,'group','Pooled descriptive — 18 different clinical scenarios')
assert len(per_antibiotic)==18 and int(per_antibiotic.n_tested.sum())==len(use)
assert int(per_antibiotic.n_above_r.sum())==int(use.observed_exceeds_r.sum())

print('OVERALL — across 18 distinct antibiotic-specific thresholds (descriptive; can mask variation):')
display(all_18[['n_tested','n_above_r','observed_exceedance_rate','mean_predicted_exceedance_probability',
                'brier_score','log_loss','auroc','n_prob_ge_90','observed_rate_among_prob_ge_90']].round(4))
print('ANTIBIOTIC-SPECIFIC results:')
display(per_antibiotic[['antibiotic','n_tested','n_above_r','observed_exceedance_rate',
                         'mean_predicted_exceedance_probability','brier_score','log_loss',
                         'auroc','average_precision','n_prob_ge_90','observed_rate_among_prob_ge_90',
                         'n_prob_le_10','observed_nonexceed_rate_among_prob_le_10',
                         'has_30_positive_and_30_negative']].round(4))
print('AUC/AP and high-confidence fractions in small strata require special caution.')

In [ ]:
#@title Cell 54.7 — Reliability tables (frequency vs predicted probability), with binomial intervals
# Fixed probability bins allow qualitative assessment of calibration. Small/empty bins should not be over-interpreted.

def wilson_95(x,n):
    if n==0:return (np.nan,np.nan)
    z=1.959963984540054
    p=x/n
    d=1+z*z/n
    center=(p+z*z/(2*n))/d
    radius=z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return float(max(0,center-radius)),float(min(1,center+radius))


def reliability_bins(t, group_name):
    bins=np.linspace(0,1,11)
    codes=np.minimum(np.floor(np.clip(t.prob_mic_exceeds_r.to_numpy(float),0,1)*10).astype(int),9)
    y=t.observed_exceeds_r.to_numpy(dtype=int)
    q=t.prob_mic_exceeds_r.to_numpy(float)
    rows=[]
    for j in range(10):
        mask=codes==j
        n=int(mask.sum());nr=int(y[mask].sum())
        lo,hi=wilson_95(nr,n)
        rows.append(dict(group=group_name, bin_low=float(bins[j]),bin_high=float(bins[j+1]),
                         n=n,n_exceeds_r=nr,
                         mean_probability=float(q[mask].mean()) if n else np.nan,
                         observed_exceedance_rate=float(nr/n) if n else np.nan,
                         wilson_lower95=lo,wilson_upper95=hi))
    return pd.DataFrame(rows)

reliability = pd.concat(
    [reliability_bins(use,'all_18_descriptive')]
    + [reliability_bins(t,abx) for abx,t in use.groupby('antibiotic',sort=True)],
    ignore_index=True)
assert len(reliability)==19*10
assert reliability.loc[reliability.group.eq('all_18_descriptive'),'n'].sum()==len(use)
print('Reliability bins — all 18 (descriptive):')
display(reliability.loc[reliability.group.eq('all_18_descriptive')].round(3))
print('Gentamicin and cefepime reliability bins:')
display(reliability.loc[reliability.group.isin(['gentamicin','cefepime']) & reliability.n.gt(0)].round(3))

In [ ]:
#@title Cell 54.8 — Diagnostic calibration figure (not yet a manuscript figure)
# Separate antibiotic panels prevent pooled calibration from hiding antibiotic-specific error patterns.
fig,axes=plt.subplots(1,3,figsize=(12.0,4.0),sharex=True,sharey=True)
PLOT_GROUPS=[('all_18_descriptive','All 18 scenarios (descriptive)'),
             ('cefepime','Cefepime'),('gentamicin','Gentamicin')]
for ax,(key,title) in zip(axes,PLOT_GROUPS):
    tab=reliability.loc[reliability.group.eq(key)&reliability.n.ge(10)].copy()
    ax.plot([0,1],[0,1],linestyle='--',linewidth=0.8,color='0.55')
    if len(tab):
        ax.errorbar(tab.mean_probability,tab.observed_exceedance_rate,
                    yerr=np.vstack([tab.observed_exceedance_rate-tab.wilson_lower95,
                                    tab.wilson_upper95-tab.observed_exceedance_rate]),
                    fmt='o',color='black',ecolor='0.50',capsize=2,markersize=4,linewidth=0.8)
        for row in tab.itertuples():
            ax.annotate(str(row.n),(row.mean_probability,row.observed_exceedance_rate),
                        xytext=(2,3),textcoords='offset points',fontsize=7)
    ax.set_title(title,fontsize=10)
    ax.set_xlim(0,1);ax.set_ylim(0,1)
    ax.set_xlabel('Mean predicted probability')
    ax.grid(color='0.9',linewidth=0.5)
axes[0].set_ylabel('Observed exceedance fraction')
fig.tight_layout()
plot_file=OUT/'54_BMCN_Breakpoint_Probability_Calibration_Diagnostic.png'
fig.savefig(plot_file,dpi=250,bbox_inches='tight')
plt.show()
print('Saved diagnostic figure:',plot_file)
print('Dots: bins with n ≥ 10; 95% Wilson intervals; numbers show bin counts.')
print('Interpret indication-specific thresholds as NB53 scenarios, not universal resistance calls.')

In [ ]:
#@title Cell 54.9 — Save reproducible outputs and provenance; final integrity audit
# All files use new NB54 names under NB54's own folder.
outputs={
    '54_BMCN_Breakpoint_Probabilities_HeldOut.csv.gz':use,
    '54_BMCN_Probability_Per_Antibiotic.csv':per_antibiotic,
    '54_BMCN_Probability_Overall_Descriptive.csv':all_18,
    '54_BMCN_Probability_Reliability_Bins.csv':reliability,
    '54_BMCN_Reconstructed_Posterior_Audit.csv':posterior_audit,
}
for filename,table in outputs.items():
    path=OUT/filename
    table.to_csv(path,index=False,compression='gzip' if filename.endswith('.gz') else None)
    print('Saved:',path,'|',len(table),'rows')

metadata={
    'notebook':'NB54 v0.1',
    'scientific_question':'Held-out posterior predictive probability log2(MIC) > log2(EUCAST R cutoff)',
    'source_paths':{key:str(val) for key,val in PATHS.items()},
    'source_sha256':{key:file_hash(val) for key,val in PATHS.items()},
    'fit_specification':'Exact NB41 BMC-N: 4501-point log-sigma grid from 0.01 to 16; b~N(0,1), sigma~Exponential(rate=0.5); b integrated analytically',
    'prediction_inputs':'Held-out Model C prediction; antibiotic-specific cutoff; training-only residual posterior. No held-out observed MIC used in probability computation.',
    'outcome_definition':'observed log2 MIC > log2(R cutoff mg/L)',
    'breakpoint_source':'NB53; EUCAST v16.1 scenario-dependent MIC cutoffs; NOT universally valid for unknown indications',
    'n_test':int(len(use)),'n_antibiotics':18,
    'n_fits':len(FITS),
    'posterior_reproduction_max_abs_quantile_cdf_error':float(posterior_audit.max_quantile_cdf_error.max()),
    'clinical_limitations':'Indication unspecified; TMP-SMX component units unverified; piperacillin-tazobactam ATU; some groups very small',
    'calibration_note':'Pooled calibration across 18 antibiotics is descriptive; inspect individual antibiotics and support counts. High probability is not automatically a validated clinical decision.',
}
provenance_file=OUT/'54_Provenance_And_Validation.json'
provenance_file.write_text(json.dumps(metadata,indent=2),encoding='utf-8')
print('Saved:',provenance_file)
assert (OUT/'54_BMCN_Breakpoint_Probabilities_HeldOut.csv.gz').is_file()
assert (OUT/'54_BMCN_Probability_Per_Antibiotic.csv').is_file()
assert (OUT/'54_BMCN_Breakpoint_Probability_Calibration_Diagnostic.png').is_file()
print('NB54 COMPLETE. Manuscript v0.6, NB40, NB41, NB53 and Supplementary Table S1 remain unchanged.')